# **The Chat Format**

In this notebook, you will explore how you can utilize the chat format to have extended conversations with chatbots personalized or specialized for specific tasks or behaviors.

## Setup

In [42]:
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [43]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

def get_completion(prompt, model="gpt-3.5-turbo", temperature=0):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    return response.choices[0].message.content


def get_completion_from_messages(message, model="gpt-3.5-turbo", temperature=0):
    response = client.chat.completions.create(
        model=model,
        messages=message,
        temperature=temperature,
    )
    return response.choices[0].message.content

In [44]:
messages =  [
{'role':'system', 'content':'You are an assistant that speaks like Shakespeare.'},
{'role':'user', 'content':'tell me a joke'},
{'role':'assistant', 'content':'Why did the chicken cross the road'},
{'role':'user', 'content':'I don\'t know'}  ]

In [45]:
response = get_completion_from_messages(messages, temperature=1)
print(response)

To get to the other side, of course! Oh, what a jest!


In [46]:
messages =  [
{'role':'system', 'content':'You are friendly chatbot.'},
{'role':'user', 'content':'Hi, my name is Isa'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

Hello Isa! It's nice to meet you. How can I assist you today?


In [47]:
messages =  [
{'role':'system', 'content':'You are friendly chatbot.'},
{'role':'user', 'content':'Yes,  can you remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

Sorry, I don't have the ability to remember personal information about users.


In [48]:
messages =  [
{'role':'system', 'content':'You are friendly chatbot.'},
{'role':'user', 'content':'Hi, my name is Isa'},
{'role':'assistant', 'content': "Hi Isa! It's nice to meet you. \
Is there anything I can help you with today?"},
{'role':'user', 'content':'Yes, you can remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

Your name is Isa!


# OrderBot
We can automate the collection of user prompts and assistant responses to build a  OrderBot. The OrderBot will take orders at a pizza restaurant.

In [49]:
def collect_messages(_):
    prompt = inp.value_input
    inp.value = ''
    context.append({'role':'user', 'content':f"{prompt}"})
    response = get_completion_from_messages(context)
    context.append({'role':'assistant', 'content':f"{response}"})
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600, styles={'background-color': '#F6F6F6'})))

    return pn.Column(*panels)


In [50]:
import panel as pn  # GUI
pn.extension()

panels = [] # collect display

context = [ {'role':'system', 'content':"""
You are OrderBot, an automated service to collect orders for a pizza restaurant. \
You first greet the customer, then collects the order, \
and then asks if it's a pickup or delivery. \
You wait to collect the entire order, then summarize it and check for a final \
time if the customer wants to add anything else. \
If it's a delivery, you ask for an address. \
Finally you collect the payment.\
Make sure to clarify all options, extras and sizes to uniquely \
identify the item from the menu.\
You respond in a short, very conversational friendly style. \
The menu includes \
pepperoni pizza  12.95, 10.00, 7.00 \
cheese pizza   10.95, 9.25, 6.50 \
eggplant pizza   11.95, 9.75, 6.75 \
fries 4.50, 3.50 \
greek salad 7.25 \
Toppings: \
extra cheese 2.00, \
mushrooms 1.50 \
sausage 3.00 \
canadian bacon 3.50 \
AI sauce 1.50 \
peppers 1.00 \
Drinks: \
coke 3.00, 2.00, 1.00 \
sprite 3.00, 2.00, 1.00 \
bottled water 5.00 \
"""} ]  # accumulate messages


inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=300),
)

dashboard

Column
    [0] TextInput(placeholder='Enter text here…')
    [1] Row
        [0] Button(name='Chat!')
    [2] ParamFunction(function, _pane=Column, defer_load=False, height=300, loading_indicator=True)

In [51]:
messages =  context.copy()
messages.append(
{'role':'system', 'content':'create a json summary of the previous food order. Itemize the price for each item\
 The fields should be 1) pizza, include size 2) list of toppings 3) list of drinks, include size   4) list of sides include size  5)total price '},
)
 #The fields should be 1) pizza, price 2) list of toppings 3) list of drinks, include size include price  4) list of sides include size include price, 5)total price '},

response = get_completion_from_messages(messages, temperature=0)
print(response)

{
  "pizza": {
    "type": "pepperoni pizza",
    "size": "large"
  },
  "toppings": [
    "extra cheese",
    "mushrooms"
  ],
  "drinks": [
    {
      "type": "coke",
      "size": "medium"
    }
  ],
  "sides": [
    {
      "type": "fries",
      "size": "regular"
    }
  ],
  "total price": 23.45
}


## Try experimenting on your own!

You can modify the menu or instructions to create your own orderbot!

# Exercise
 - Complete the prompts similar to what we did in class.
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [55]:
import panel as pn  # GUI
pn.extension()

panels = []  # Collect display


# CoffeeBot Context
context_coffee = [ {'role':'system', 'content':"""
You are CoffeeBot, an automated assistant for a coffee shop. You greet the customer,
take their coffee order, confirm size and any customizations, then ask if they want
food or pastries. Summarize the order before finalizing payment.
"""} ]

def collect_messages_coffee(_):
    prompt = inp_coffee.value
    inp_coffee.value = ''
    context_coffee.append({'role': 'user', 'content': f"{prompt}"})
    response = get_completion_from_messages(context_coffee)
    context_coffee.append({'role': 'assistant', 'content': f"{response}"})
    panels.append(pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(pn.Row('Assistant:', pn.pane.Markdown(response, width=600, styles={'background-color': '#F6F6F6'})))
    return pn.Column(*panels)

inp_coffee = pn.widgets.TextInput(value="Hi", placeholder='Enter your coffee order…')
button_coffee = pn.widgets.Button(name="Order Coffee!")
interactive_coffee = pn.bind(collect_messages_coffee, button_coffee)

dashboard_coffee = pn.Column(
    inp_coffee,
    pn.Row(button_coffee),
    pn.panel(interactive_coffee, loading_indicator=True, height=300),
)


dashboard_coffee


Column
    [0] TextInput(placeholder='Enter your coffee o...)
    [1] Row
        [0] Button(name='Order Coffee!')
    [2] ParamFunction(function, _pane=Column, defer_load=False, height=300, loading_indicator=True)

In [56]:
import panel as pn  # GUI
pn.extension()

panels = []  # Collect display


# SushiBot Context
context_sushi = [ {'role':'system', 'content':"""
You are SushiBot, an automated assistant for a sushi restaurant. You greet the customer,
take their sushi order, confirm the type of rolls or sashimi, then ask if they want
any sides or drinks. Summarize the order before finalizing payment.
"""} ]

def collect_messages_sushi(_):
    prompt = inp_sushi.value
    inp_sushi.value = ''
    context_sushi.append({'role': 'user', 'content': f"{prompt}"})
    response = get_completion_from_messages(context_sushi)
    context_sushi.append({'role': 'assistant', 'content': f"{response}"})
    panels.append(pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(pn.Row('Assistant:', pn.pane.Markdown(response, width=600, styles={'background-color': '#F6F6F6'})))
    return pn.Column(*panels)

inp_sushi = pn.widgets.TextInput(value="Hi", placeholder='Enter your sushi order…')
button_sushi = pn.widgets.Button(name="Order Sushi!")
interactive_sushi = pn.bind(collect_messages_sushi, button_sushi)

dashboard_sushi = pn.Column(
    inp_sushi,
    pn.Row(button_sushi),
    pn.panel(interactive_sushi, loading_indicator=True, height=300),
)


dashboard_sushi

Column
    [0] TextInput(placeholder='Enter your sushi order…')
    [1] Row
        [0] Button(name='Order Sushi!')
    [2] ParamFunction(function, _pane=Column, defer_load=False, height=300, loading_indicator=True)

In [57]:
import panel as pn  # GUI
pn.extension()

panels = []  # Collect display

# MovieSnackBot Context
context_moviesnack = [ {'role':'system', 'content':"""
You are MovieSnackBot, an automated assistant for a movie theater snack stand. You greet
the customer, take their snack order, confirm size and extras, then ask if they want drinks.
Summarize the order before finalizing payment.
"""} ]

def collect_messages_moviesnack(_):
    prompt = inp_moviesnack.value
    inp_moviesnack.value = ''
    context_moviesnack.append({'role': 'user', 'content': f"{prompt}"})
    response = get_completion_from_messages(context_moviesnack)
    context_moviesnack.append({'role': 'assistant', 'content': f"{response}"})
    panels.append(pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(pn.Row('Assistant:', pn.pane.Markdown(response, width=600, styles={'background-color': '#F6F6F6'})))
    return pn.Column(*panels)

inp_moviesnack = pn.widgets.TextInput(value="Hi", placeholder='Enter your movie snack order…')
button_moviesnack = pn.widgets.Button(name="Order Snacks!")
interactive_moviesnack = pn.bind(collect_messages_moviesnack, button_moviesnack)

dashboard_moviesnack = pn.Column(
    inp_moviesnack,
    pn.Row(button_moviesnack),
    pn.panel(interactive_moviesnack, loading_indicator=True, height=300),
)

dashboard_moviesnack


Column
    [0] TextInput(placeholder='Enter your movie s...)
    [1] Row
        [0] Button(name='Order Snacks!')
    [2] ParamFunction(function, _pane=Column, defer_load=False, height=300, loading_indicator=True)

**Evaluating AI OrderBot Variations**

**Summary of Findings**
Through experimentation with three different AI OrderBots (CoffeeBot, SushiBot, and MovieSnackBot), we observed how structured prompts help shape chatbot behavior. Each variation had a unique personality and specialized in collecting specific types of orders while keeping conversations interactive and engaging.

**Performance of Each Bot**
1- CoffeeBot performed well in guiding users through coffee orders. It effectively confirmed size and customization preferences. However, it sometimes failed to suggest food options naturally. Adding a rule to encourage upselling could improve this.

2- SushiBot excelled at breaking down sushi orders into clear choices (e.g., roll type, sides). However, it struggled with uncommon sushi requests like omakase (chef’s choice). A better version would include dynamic responses for less standard orders.

3- MovieSnackBot was the most successful in its domain. It naturally guided customers through popcorn sizes and additional purchases. However, it sometimes assumed users wanted a combo deal when they only requested a single item. Adjusting the flow to confirm preferences before offering a combo would improve user experience.



**What Didn’t Work Well?**

Over-Assumptions: Some bots assumed users wanted add-ons, which could lead to frustrating interactions.
Limited Memory: The AI sometimes lost track of previous responses, requiring users to repeat selections.
Rigid Responses: If a user requested something unusual (e.g., a "half-caf oat milk cappuccino"), the AI struggled unless explicitly programmed for it.


**Key Learnings**

- Well-Defined Prompts Matter: A well-structured system prompt significantly impacts chatbot performance and flow.
Handling Uncommon Requests: AI order-taking should be flexible enough to process unique or unexpected orders.
- Natural Upselling Improves Experience: Bots that gently offer suggestions (rather than assume) perform better.
- Memory and Context Tracking: A chatbot remembering past inputs enhances customer interactions.

**Conclusion**
Overall, all variations of OrderBot were functional and effective in their respective domains. Future improvements could include refining upselling strategies, handling special requests, and integrating memory for a smoother conversational experience.